# Keyword Extraction via LLM

Extracts technical keywords from each article's TEI XML using a local LLM (Qwen3-14B via vLLM). Each article is processed in multiple passes: one call on the abstract, then one call per body chunk. Results are saved incrementally as checkpoints every 50 files.

**Input:** `output/xml_english/` — TEI XML files from `PDFtoXML.ipynb`  
**Output:**
- `keywords_results/keywords_checkpoint_N.csv` — intermediate checkpoints
- `keywords_results/keywords_FINAL.csv` — results for the last batch
- `keywords_results/keywords_ALL_MERGED.csv` — all batches merged (input for `dbscan.ipynb`)

**Prerequisites:** vLLM server must be running:
```
vllm serve Qwen/Qwen3-14B-AWQ --port 8000
```

In [1]:
import logging
import sys
import time
import xml.etree.ElementTree as ET
from concurrent.futures import ThreadPoolExecutor
from datetime import datetime
from pathlib import Path

import pandas as pd
from langchain_openai import ChatOpenAI
from langchain_text_splitters import TokenTextSplitter
from pydantic import BaseModel, Field
from tqdm import tqdm


In [ ]:
PROJECT_ROOT = Path("..").resolve()

XML_DIR      = PROJECT_ROOT / "output" / "xml_english"   # TEI XML input
KEYWORDS_DIR = PROJECT_ROOT / "keywords_results"         # checkpoints + final CSV

# Local LLM server (vLLM)
LLM_BASE_URL  = "http://0.0.0.0:8000/v1"
LLM_MODEL     = "Qwen/Qwen3-14B-AWQ"

# Text splitting
CHUNK_SIZE    = 1000   # tokens per body chunk sent to the LLM
CHUNK_OVERLAP = 100
MAX_WORKERS   = 10    # parallel LLM calls per file

logging.basicConfig(
    format="%(asctime)s | %(levelname)s : %(message)s",
    level=logging.INFO,
    stream=sys.stdout,
)
logger = logging.getLogger(__name__)


In [2]:
class ExtractedArticleParts(BaseModel):
    title: str | None = None
    introduction: str | None = None
    abstract: str | None = None
    body: str | None = None

class Keyword(BaseModel):
    keyword_string: str = Field(description="The keyword string.")
    keyword_strength: int = Field(ge=1, le=5, description="The strength of the keyword from 1 to 5")


class ResponseSchema(BaseModel):
    keywords: list[Keyword] = Field(
        description="List of up to 30 extracted keywords from the given text.",
        min_items=5,
        max_items=30
    )


/tmp/ipykernel_3789/7913318.py:13: PydanticDeprecatedSince20: `min_items` is deprecated and will be removed, use `min_length` instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.12/migration/
  keywords: list[Keyword] = Field(
/tmp/ipykernel_3789/7913318.py:13: PydanticDeprecatedSince20: `max_items` is deprecated and will be removed, use `max_length` instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.12/migration/
  keywords: list[Keyword] = Field(


In [3]:
SYSTEM_MESSAGE = """############################
You are a keyword extraction specialist for academic articles on computer chess.
Your task is to identify and return the most relevant technical keywords from the given article.

## What to extract
- Technical terms specific to computer chess and chess AI: search algorithms, board representations, data structures, evaluation techniques, engine architecture concepts.
- Machine learning and AI terms when they appear in a chess-related context: neural architectures, training methods, learning paradigms.
- Game-theoretic and complexity concepts directly tied to chess game-tree analysis.
- Broader computer science terms only when the article uses them in a computer-chess context.
- If the article describes a well-known technique without naming it, infer and return the canonical technical term.

## What to exclude
- Generic academic words that are not technical terms (e.g., "result", "method", "approach", "study", "performance", "analysis").
- Chess rules, openings/defenses by name, historical facts, player names, tournament names.
- Vague or overly broad terms (e.g., "artificial intelligence", "computer science") unless the paper specifically focuses on them.

## Keyword strength
For each keyword, assign `keyword_strength` from 1 to 5:
- 5: central/core concept of the article
- 4: highly important supporting concept
- 3: clearly relevant technical concept
- 2: minor but still meaningful technical concept
- 1: weak/occasional relevance


## Formatting rules
- Use lowercase.
- Use singular form ("neural network", not "neural networks").
- Prefer the most widely recognized name for each concept.
- Return JSON in this exact format:
{{
  "keywords": [
    {"keyword_string": "keyword1", "keyword_strength": 5},
    {"keyword_string": "keyword2", "keyword_strength": 3}
  ]
}}
############################
"""

USER_MESSAGE = """
Extract the most relevant computer-chess and chess-AI keywords from the article below.
For each keyword, also return `keyword_strength` from 1 (weak relevance) to 5 (core concept).

##############
ARTICLE:
<article>
<title>
{title}
</title>

<text>
{text}
</<text>

</article>
############################

Respond with a JSON object in this exact format:
{{
  "keywords": [
    {{"keyword_string": "keyword1", "keyword_strength": 5}},
    {{"keyword_string": "keyword2", "keyword_strength": 3}}
  ]
}}
"""

In [ ]:
class KeywordExtractor:
    """
    Extracts keywords from TEI XML files using a local LLM.

    Strategy per article:
      1. title + abstract  (one LLM call)
      2. title + body chunk 1, title + body chunk 2, ...  (one call per chunk)
    All calls for a single file are executed in parallel.
    Checkpoints are saved every 50 files so progress is not lost on failure.
    """

    def __init__(self, xml_dir: Path, output_dir: Path = KEYWORDS_DIR):
        self.xml_dir = Path(xml_dir)
        self.output_dir = Path(output_dir)
        self.output_dir.mkdir(exist_ok=True, parents=True)
        self.text_splitter = TokenTextSplitter(
            encoding_name="gpt2",
            chunk_size=CHUNK_SIZE,
            chunk_overlap=CHUNK_OVERLAP,
        )

    def extract_text_from_xml(self, xml_file: Path) -> ExtractedArticleParts | None:
        """Extract title, abstract, introduction, and body text from a TEI XML file."""
        try:
            root = ET.parse(xml_file).getroot()
            ns = {"tei": "http://www.tei-c.org/ns/1.0"}
            parts = ExtractedArticleParts()

            title = root.find(".//tei:titleStmt/tei:title", ns)
            if title is not None and title.text:
                parts.title = title.text.strip()

            abstract = root.find(".//tei:abstract", ns)
            if abstract is not None:
                text = "".join(abstract.itertext()).strip()
                if len(text) > 50:
                    parts.abstract = text

            body = root.find(".//tei:body", ns)
            if body is not None:
                for div in body.findall(".//tei:div", ns):
                    head = div.find(".//tei:head", ns)
                    if head is not None and head.text and (
                        "introduction" in head.text.lower()
                        or "overview" in head.text.lower()
                    ):
                        intro = "".join(div.itertext()).strip()
                        if len(intro) > 100:
                            parts.introduction = (
                                intro if parts.introduction is None
                                else parts.introduction + "\n" + intro
                            )

                paragraphs = [
                    "".join(p.itertext()).strip()
                    for p in body.findall(".//tei:p", ns)
                    if len("".join(p.itertext()).strip()) > 50
                ]
                if paragraphs:
                    parts.body = "\n".join(paragraphs)

            return parts
        except Exception as e:
            logger.error("Failed to extract text from %s: %s", xml_file.name, e)
            return None

    def split_body_into_chunks(self, body: str) -> list[str]:
        """
        Split body text into token-bounded chunks for LLM processing.
        TokenTextSplitter is used (over character-based) to respect the model context window.
        """
        return self.text_splitter.split_text(body) if body.strip() else []

    def _call_llm(self, llm: ChatOpenAI, title: str, text: str) -> list[dict]:
        """Make a single LLM call and return a list of {keyword_string, keyword_strength} dicts."""
        messages = [
            {"role": "system", "content": SYSTEM_MESSAGE},
            {"role": "user",   "content": USER_MESSAGE.format(title=title, text=text)},
        ]
        response = llm.invoke(messages)
        return [kw.model_dump() for kw in ResponseSchema.model_validate_json(response.content).keywords]

    def process_file_chunks(self, xml_file: Path, llm: ChatOpenAI) -> list[list[dict]]:
        """
        Process one XML file: run title+abstract and title+body_chunk LLM calls in parallel.
        Returns a list of keyword lists (one per LLM call).
        """
        parts = self.extract_text_from_xml(xml_file)
        if not parts:
            return []

        title = parts.title or ""
        if not title:    
            logger.warning("No title found in %s", xml_file.name)
        if not parts.abstract: 
            logger.warning("No abstract found in %s", xml_file.name)
        if not parts.body:     
            logger.warning("No body found in %s", xml_file.name)

        texts: list[str] = []
        if parts.abstract and parts.abstract.strip():
            texts.append(parts.abstract)
        texts.extend(self.split_body_into_chunks(parts.body or ""))

        if not texts:
            return []

        with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
            return list(executor.map(lambda t: self._call_llm(llm, title, t), texts))

    def process_all(self, llm: ChatOpenAI, start_from: int = 0,
                    max_files: int | None = None) -> tuple[list[dict], dict]:
        """
        Run keyword extraction on all XML files in xml_dir.

        Args:
            llm:        Configured ChatOpenAI instance.
            start_from: Skip the first N files (resume from a checkpoint).
            max_files:  Process at most this many files (useful for testing).

        Returns:
            (results, stats) where results is a list of per-file dicts.
        """
        xml_files = sorted(self.xml_dir.glob("*.tei.xml"))
        total_all = len(xml_files)
        xml_files = xml_files[start_from: start_from + max_files] if max_files else xml_files[start_from:]

        stats = {"total": len(xml_files), "processed": 0, "skipped": 0, "errors": 0}
        results: list[dict] = []

        print(f"Keyword extraction | files: {len(xml_files)} (total in dir: {total_all})")
        print(f"Output: {self.output_dir}")
        print(f"Start: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
        start_time = time.time()

        with tqdm(total=len(xml_files), unit="file", ncols=100) as pbar:
            for i, xml_file in enumerate(xml_files, start_from + 1):
                try:
                    kw_lists = self.process_file_chunks(xml_file, llm)
                    if not kw_lists:
                        stats["skipped"] += 1
                    else:
                        results.append({
                            "filename": xml_file.stem.replace(".grobid.tei", ""),
                            "n_chunks": len(kw_lists),
                            "keywords_lists": kw_lists,
                        })
                        stats["processed"] += 1
                    if i % 50 == 0:
                        self.save_checkpoint(results, i)
                        tqdm.write(f"Checkpoint saved: {i}/{total_all}")
                except Exception as e:
                    stats["errors"] += 1
                    tqdm.write(f"Error {xml_file.name}: {str(e)[:50]}")
                finally:
                    pbar.update(1)

        self.save_final(results)
        self.print_statistics(results, stats, start_time)
        return results, stats

    def save_checkpoint(self, results: list[dict], batch: int, error: bool = False) -> None:
        """Save intermediate results to a numbered checkpoint CSV."""
        if not results:
            return
        suffix = "_ERROR" if error else ""
        path = self.output_dir / f"keywords_checkpoint_{batch}{suffix}.csv"
        pd.DataFrame(results).to_csv(path, index=False, encoding="utf-8")

    def save_final(self, results: list[dict]) -> Path | None:
        """Save final extraction results to keywords_FINAL.csv."""
        if not results:
            print("No results to save")
            return None
        path = self.output_dir / "keywords_FINAL.csv"
        pd.DataFrame(results).to_csv(path, index=False, encoding="utf-8")
        print(f"\nSaved {len(results)} records -> {path}")
        return path

    def print_statistics(self, results: list[dict], stats: dict, start_time: float) -> None:
        """Print a summary of extraction results."""
        elapsed = time.time() - start_time
        print(f"Total:     {stats['total']}")
        print(f"Processed: {stats['processed']} ({stats['processed'] / stats['total'] * 100:.1f}%)")
        print(f"Skipped:   {stats['skipped']}")
        print(f"Errors:    {stats['errors']}")
        if results:
            chunks = [r['n_chunks'] for r in results]
            print(f"Chunks/file: avg={sum(chunks)/len(chunks):.1f} min={min(chunks)} max={max(chunks)}")
        print(f"Time: {time.strftime('%H:%M:%S', time.gmtime(elapsed))}")
        if elapsed > 0 and stats['processed'] > 0:
            print(f"Speed: {stats['processed'] / elapsed:.2f} files/sec")


In [ ]:
llm_guided = ChatOpenAI(
    base_url=LLM_BASE_URL,
    api_key="",  # not required for local vLLM server
    model_name=LLM_MODEL,
    extra_body={
        "structured_outputs": {"json": ResponseSchema.model_json_schema()},
        "chat_template_kwargs": {"enable_thinking": False},
    },
    temperature=0,
)

In [ ]:
extractor = KeywordExtractor(xml_dir=XML_DIR, output_dir=KEYWORDS_DIR)

# To resume from a checkpoint, set start_from to the last saved checkpoint number.
# Example: start_from=850 skips the first 850 files already saved in checkpoint_850.csv
results, stats = extractor.process_all(llm=llm_guided, start_from=0)

## Merging checkpoints

If processing was split across multiple runs, merge the checkpoint CSVs into one file.
Adjust the filenames below to match your actual checkpoint files.

In [ ]:
# Collect all checkpoint files and the final batch, then merge into one CSV.
# This works regardless of how many batches the run was split into.
checkpoint_files = sorted(KEYWORDS_DIR.glob("keywords_checkpoint_*.csv"),
                           key=lambda p: int(p.stem.split('_')[-1]))
final_file = KEYWORDS_DIR / "keywords_FINAL.csv"

dfs = [pd.read_csv(f) for f in checkpoint_files]
if final_file.exists():
    dfs.append(pd.read_csv(final_file))

df_all = pd.concat(dfs, ignore_index=True)
df_all.to_csv(KEYWORDS_DIR / "keywords_ALL_MERGED.csv", index=False)
print(f"Merged {len(checkpoint_files)} checkpoint(s) + final batch")
print(f"Total articles: {len(df_all)} -> {KEYWORDS_DIR / 'keywords_ALL_MERGED.csv'}")


## Coverage check (optional)

Identify XML files that were skipped or failed during extraction.

In [ ]:
df_merged = pd.read_csv(KEYWORDS_DIR / "keywords_ALL_MERGED.csv")
processed = set(df_merged["filename"].tolist())

all_files = {f.stem.replace(".grobid.tei", "") for f in XML_DIR.glob("*.tei.xml")}
missing = all_files - processed

print(f"Processed: {len(processed)} / {len(all_files)}")
print(f"Missing:   {len(missing)}")
for f in sorted(missing):
    print(f"  - {f}")

Skpped: 10
 - 01FierkeNicholson
 - 04Braunlich
 - 17AndersonGreen
 - 23Feng
 - 25Hwang strategic reasoning
 - 50Turing Computing machines and intelligence
 - 53Slater Statistics
 - 63Kavka
 - 73MarslandRushton
 - 89Tesauro connectionist-learning-of-expert-preferences-by-comparison-training
